# Object Enrichments in ApertureDB: Questions You Can Now Ask

You have a bucket of images (here 2,560 movie posters). Google Cloud Storage's **object enrichments** describe each one: labels, objects, faces and content-safety ratings.
`ingest.py` and `enrich.py` loaded the images and those descriptions into ApertureDB as a graph. This notebook asks the questions people actually have of such a library.
Each section starts with the question in plain words; **the query that answers it is underneath, collapsed: click "Query" to open it.**

**Contents:** 1 What Did Google Find? - 2 Find the Right Posters for a Brief - 3 Family Friendly: Choose, Hold Back, Explain - 4 Find Posters by Words - 5 Find the Right Faces - 6 Follow the Connections - 7 See Why a Poster Matched - 8 What Overlaps What - 9 How Sure Are the Objects? - 10 Add Vector Search

```
 _Image ──HasLabel──────▶ GCSLabel        50 whole-image labels with a score
   ▲  └──HasDetection──▶ _BoundingBox      same boxes, with the edge properties detector / score / stored_from
   └───── _BoundingBox                    object boxes (named by the object) and face boxes (label "face"),
                                          each with its own properties (score, joy, pan angle, ...)
```

Sections 1-9 use only the enrichments (no vectors); section 10 adds vector search. The same queries are saved as JSON in `queries/` so you can paste them into the ApertureDB web UI.
Run `./setup.sh` and `./demo.sh --bucket ... --prefix ...` first (see the README).

## Connect

In [ ]:
import html
import json
import os

import pandas as pd
from IPython.display import HTML, display

import config          # which ApertureDB: local Community Edition, or APERTUREDB_KEY / .env
import graph_viz as gv  # interactive graph views
import queries         # every query below, as plain functions returning ApertureDB queries

client = config.get_client()

DATASET = None  # set to "<bucket>/<prefix>" to restrict every query to one ingested folder
LABELS = {"_Image": "gcs_object_stem", "GCSLabel": "description"}   # how graph nodes are named
BY_SCORE = {"GCSLabel": ("score", "descending")}                    # keep the best labels first

In [ ]:
def run(query):
    """Run a query; raise if the server rejects a command; return (response, blobs)."""
    response, blobs = client.query(query)
    if isinstance(response, dict):
        raise RuntimeError(response.get("info"))
    for r in response:
        body = next(iter(r.values()))
        if body.get("status", 0) not in (0, 2):
            raise RuntimeError(body.get("info"))
    return response, blobs


def table(response, index=-1, columns=None):
    """The entities one command returned, as a DataFrame."""
    body = next(iter(response[index].values()))
    if "groups" in body:  # results.group: one row per group, with its count (and any avg/min/max)
        df = pd.DataFrame(body["groups"]).drop(columns="_group_class", errors="ignore")
        if "_group_avg" in df:
            df["avg_score"] = df.pop("_group_avg").map(lambda d: round(d["score"], 2))
        return df.rename(columns={"_group_count": "count"}).sort_values("count", ascending=False, ignore_index=True)
    rows = body.get("entities", [])
    if isinstance(rows, dict):  # grouped by source
        rows = [e for group in rows.values() for e in group]
    df = pd.DataFrame(rows)
    return df if columns is None or df.empty else df[[c for c in columns if c in df]]


def thumbs(blobs, captions, width=170):
    """Show returned pixels (posters or box crops) in a row."""
    import base64
    cells = "".join(
        f'<figure style="display:inline-block;margin:4px;text-align:center;font:12px sans-serif">'
        f'<img width="{width}" src="data:{gv.sniff_mime(b)};base64,{base64.b64encode(b).decode()}">'
        f'<figcaption>{html.escape(str(c))}</figcaption></figure>' for b, c in zip(blobs, captions))
    display(HTML(cells))


def compact(obj, indent=0, width=96):
    """JSON text that keeps short pieces on one line, so a query stays readable."""
    flat = json.dumps(obj)
    if len(flat) + indent <= width or not isinstance(obj, (dict, list)):
        return flat
    pad = " " * (indent + 2)
    if isinstance(obj, dict):
        items = [f"{pad}{json.dumps(k)}: {compact(v, indent + 2, width)}" for k, v in obj.items()]
        return "{\n" + ",\n".join(items) + "\n" + " " * indent + "}"
    return "[\n" + ",\n".join(pad + compact(v, indent + 2, width) for v in obj) + "\n" + " " * indent + "]"


def show(query, caption=None, columns=None, top=None):
    """Print the query, run it, show the rows (the first ``top``) and - if it returned pixels - the pictures."""
    display(HTML("<details><summary><b>Query</b> (ApertureDB JSON): click to open</summary><pre>"
                 + html.escape(compact(query)) + "</pre></details>"))
    response, blobs = run(query)
    df = table(response, columns=columns)
    if not len(df):
        display(HTML("<i>no rows - ingest more data or loosen the filter</i>"))
    else:
        display(df.head(top) if top else df)
        if top and len(df) > top:
            display(HTML(f"<i>showing {top} of {len(df)} rows</i>"))
    if blobs and caption and len(df):
        thumbs(blobs, [caption(r) for _, r in df.iterrows()])
    return response, blobs

## The Data Model

Live from the database: one node per class (with how many objects it holds), one edge per connection class.
Hover a class to highlight its connections, click it for its properties and a query that lists it.

In [ ]:
gv.render_schema(client)

## 1. What Did Google Find?

Before asking for anything, see what the detectors reported across the whole library. The counting happens inside the database (`results.group`): no rows to download, one row per group.

### 1.1 Which Words Can I Search For?

**The question:** I am about to write a search. Which labels exist, and how many posters carry each?

**What the answer shows:** every distinct label with its poster count and average score; use it to pick the words for the sections below.

*How:* `results.group` on `description` with `count` and `avg`, over labels scoring at least 0.5 (the server groups first, so the table is sorted here).

In [ ]:
response, _ = show(queries.label_vocabulary(min_score=0.5, dataset=DATASET), top=15)
vocabulary = table(response)
print(f"{len(vocabulary):,} distinct labels")

### 1.2 Is There an Alien, a Robot, a Ghost?

**The question:** A colleague asks whether the library has aliens (or robots, ghosts). Check before promising.

**What the answer shows:** which of those words exist as labels, and how many posters carry them. There is no *alien* label: the detector says *outer space* or *robot* instead, so a search for aliens needs more than the labels.

*How:* plain pandas over the result of 1.1; change the words.

In [ ]:
vocabulary[vocabulary.description.str.contains("alien|robot|ghost|superhero|monster", case=False)]

### 1.3 How Risky Is the Library?

**The question:** How many posters would a family catalogue have to think about?

**What the answer shows:** the number of posters for every (adult, violent) rating pair; the scale is VERY_UNLIKELY, UNLIKELY, POSSIBLE, LIKELY, VERY_LIKELY.

*How:* `results.group` on two properties at once.

In [ ]:
_ = show(queries.safety_ratings(dataset=DATASET), top=12)

## 2. Find the Right Posters for a Brief

The questions a person would actually type, answered by combining what the detectors found.

### 2.1 A Family-Friendly Sports Movie With a Team in Frame

**The question:** *"A family-friendly sports movie about a team, with three or more people in frame."* (the ask from the talk and from `demo.ipynb`)

**What the answer shows:** posters with a sports label, at least three detected faces and family-friendly ratings, most faces first.

*How:* three detectors in one request: a label search, a graph hop to the poster, and conditions on `face_count` and the two safety ranks. `demo.ipynb` adds the vector part.

In [ ]:
_ = show(queries.team_sports_movies(dataset=DATASET, limit=8),
         caption=lambda r: f"{r.face_count} faces, adult {r.adult_content_likelihood.lower()}",
         columns=["gcs_object_name", "face_count", "adult_content_likelihood", "violent_content_likelihood"])

### 2.2 Characters Who Look Happy, for a Campaign

**The question:** Marketing wants posters of fictional characters where *everyone* is smiling.

**What the answer shows:** posters labelled *fictional character* on which every detected face has joy of at least 0.7 (posters without faces never match).

*How:* a label search (`icontains`), a hop from the label to its poster, and a condition on `min_face_joy` (the least happy face on the poster, stored at load time). Try `term="cartoon"` or `"superhero"`.

In [ ]:
_ = show(queries.everyone_looks_happy(dataset=DATASET),
         caption=lambda r: f"{r.face_count} face(s), least happy {r.min_face_joy:.2f}",
         columns=["gcs_object_name", "face_count", "min_face_joy", "label_count"])

## 3. Family Friendly: Choose, Hold Back, Explain

The content-safety detector rates every poster for adult and violent content (VERY_UNLIKELY ... VERY_LIKELY). Combined with labels and faces, one request can answer a natural-language ask.

### 3.1 Family Movie Night

**The question:** A parent wants something animated the whole family can watch, where everyone looks happy.

**What the answer shows:** family-friendly (both ratings VERY_UNLIKELY) posters labelled *animation* on which every face is smiling.

*How:* a label search, a hop to the poster, then conditions on the safety ranks and on `min_face_joy`. Try `term="cartoon"` or `min_joy=0.8`.

In [ ]:
_ = show(queries.family_movie_night(dataset=DATASET),
         caption=lambda r: f"{r.face_count} face(s), least happy {r.min_face_joy:.2f}",
         columns=["gcs_object_name", "face_count", "min_face_joy", "adult_content_likelihood", "violent_content_likelihood"])

### 3.2 A Family-Safe Shelf for Any Topic

**The question:** A kids' catalogue wants a shelf for any topic (animation today, something else tomorrow) with nothing risky on it.

**What the answer shows:** posters carrying the label that are rated VERY_UNLIKELY adult **and** VERY_UNLIKELY violent.

*How:* a label search, a hop, and two rank conditions combined with `all`. Try another word from 1.1.

In [ ]:
_ = show(queries.family_friendly("animation", dataset=DATASET),
         caption=lambda r: f"{r.adult_content_likelihood} / {r.violent_content_likelihood}")

### 3.3 What Should the Family Catalogue Hold Back?

**The question:** The review team wants the posters that must not appear on the kids' shelf.

**What the answer shows:** posters with either rating at least POSSIBLE (rank 3 of 5), no pictures, just the ratings.

*How:* an `any` expression across two properties written by the same detector.

In [ ]:
_ = show(queries.sensitive_content(dataset=DATASET))

### 3.4 Why Was It Held Back?

**The question:** Someone asks why their poster is not on the kids' shelf.

**What the answer shows:** the held-back posters with their highest-scoring labels, which usually explain the rating.

*How:* two commands: the flagged images, then their connected labels sorted by score.

In [ ]:
_ = show(queries.why_flagged(dataset=DATASET), columns=['description', 'score', 'image_id'])

## 4. Find Posters by Words

A person types a word. The label text can be searched by substring, pattern or whole word (trigram and full-text indexes keep it fast).

### 4.1 Everything About Action

**The question:** An editor types *action* and wants the posters whose labels mention it.

**What the answer shows:** posters carrying a label that contains the word, with how many labels and faces each has.

*How:* a case-insensitive substring search (`icontains`), then a hop to the posters. Change `"action"` to any word from 1.1.

In [ ]:
_ = show(queries.labels_containing("action", min_score=0.6, dataset=DATASET, images=True, limit=8),
         caption=lambda r: f"{r.label_count} labels, {r.face_count} face(s)")

### 4.2 Posters With People in Them

**The question:** Which posters show a man, woman, girl or boy?

**What the answer shows:** posters with an object box named like a person, as the object detector sees them.

*How:* `regex` with `(?i)` for any case.

In [ ]:
_ = show(queries.people_by_name(dataset=DATASET, limit=10))

### 4.3 Action or Fiction, but Not Games

**The question:** An editor wants action or fiction labels, but nothing about video games.

**What the answer shows:** matching labels above a score, best first.

*How:* an array expression with `any`, `not` and a numeric condition. Try `golf` or `romance`.

In [ ]:
_ = show(queries.label_search_boolean(include=("action", "fiction"), exclude=("game",), dataset=DATASET),
         columns=["description", "score", "image_id"])

### 4.4 Typing Mistakes Are Fine

**The question:** Someone types *animatd cartoon* (a typo).

**What the answer shows:** posters whose labels still match *Animation* and *Animated cartoon*.

*How:* a `fulltext` index on the label text: whole words, any case, `max_edits` typos per word. Unlike `icontains` it does not match inside words.

In [ ]:
_ = show(queries.label_word_search("animatd cartoon", max_edits=1, dataset=DATASET), columns=["gcs_object_name", "label_count", "face_count"])

## 5. Find the Right Faces

Faces are stored as boxes with their own properties (joy, sorrow, anger, surprise, which way they look), and the crops are returned with the answer.

### 5.1 Smiling Faces Looking at the Camera

**The question:** A designer wants smiling faces that look straight at the viewer, for a mood board.

**What the answer shows:** face crops with more joy than sorrow **and** more joy than anger, facing the camera (`abs(pan_angle) < 20`).

*How:* `$joy > $sorrow` compares two properties of the same face inside the database; nothing is post-processed in Python.

In [ ]:
_ = show(queries.happy_frontal_faces(dataset=DATASET, crops=True, limit=8),
         caption=lambda r: f"joy {r.joy:.2f}", columns=["image_id", "joy", "pan_angle", "area_fraction"])

### 5.2 Delighted, Not Just Happy

**The question:** The campaign wants delight: joy mixed with surprise, no sorrow.

**What the answer shows:** faces where `joy + surprise > 0.85` while `sorrow < 0.1`.

*How:* arithmetic over properties.

In [ ]:
_ = show(queries.delighted_faces(dataset=DATASET))

### 5.3 Posters Dominated by One Face

**The question:** A poster wall wants a close-up on every tile.

**What the answer shows:** a confidently detected face that covers a large share of the poster (`area_fraction`), as crops.

*How:* numeric conditions on box properties.

In [ ]:
_ = show(queries.dominant_faces(dataset=DATASET, crops=True),
         caption=lambda r: f"{r.area_fraction:.0%} of poster", columns=["image_id", "area_fraction", "detection_confidence"])

### 5.4 Has Every Image Been Enriched?

**The question:** Before trusting any answer: did the detectors run on everything?

**What the answer shows:** how many images have no face-detector result stored yet (zero when everything is enriched).

*How:* `is_missing` on a property.

In [ ]:
response, _ = run(queries.not_yet_enriched(dataset=DATASET))
response[0]["FindImage"]["count"]

## 6. Follow the Connections

The detections are connected to their poster, so one request can walk from a label to a poster to its faces.

### 6.1 Happy Faces on Posters Tagged *Film*

**The question:** Take posters tagged *film* and show me their happy faces.

**What the answer shows:** three hops in one request: label, then poster, then its faces with joy of at least 0.3.

*How:* chained `_ref` / `is_connected_to` / `image_ref`.

In [ ]:
_ = show(queries.happy_faces_on_tagged_posters(term="film", min_joy=0.3, dataset=DATASET),
         columns=["image_id", "joy", "area_fraction"])

### 6.2 Clothing With a Person in It

**The question:** A retailer wants posters where clothing appears *and* a person was detected.

**What the answer shows:** posters connected to a *clothing* label and to a *person* object box.

*How:* `is_connected_to` with `all`.

In [ ]:
_ = show(queries.label_and_object_together("clothing", "person", dataset=DATASET))

### 6.3 Crowd Scenes

**The question:** A producer wants crowded posters: many faces and many objects.

**What the answer shows:** posters above a face count and an object count, with no joins.

*How:* `face_count` and `object_count` are stored on the image at load time.

In [ ]:
_ = show(queries.crowded_posters(min_faces=3, min_objects=5, dataset=DATASET))

### 6.4 A Second Look at Low-Confidence Objects

**The question:** The object detector's cut-off is a choice. What did it find just below it?

**What the answer shows:** object boxes that were stored but score below 0.5, found through the `score` on their `HasDetection` edge. Moving the cut-off needs no new call to Google.

*How:* `is_connected_to` with `connection_class` and edge `constraints`; widen `low` / `high` and run again.

In [ ]:
_ = show(queries.objects_between(0.3, 0.5, dataset=DATASET), columns=["object_name", "score", "image_id"])

## 7. See Why a Poster Matched

**The question:** what does the database actually hold about a poster that came back, and how do the detections hang together?

Two pictures. Click any dot for its details: a poster shows its picture with every stored box drawn on it. The numbers are poster ids; the detail panel shows the poster itself.
Each poster has its own label nodes (a label is stored per poster, with its score), so the pictures are stars, one per poster.

### 7.1 Four Sports Posters From 2.1 and Their Strongest Labels

Blue dots are the posters from 2.1; around each, the labels the detector is surest of. The label that matched the search (a sports word) may sit lower in that list: raise `per_class` to see more.

In [ ]:
gv.render_query(client, queries.team_sports_movies(dataset=DATASET, limit=4, images=False), title="sports posters and their top labels",
                label_props=LABELS, sort=BY_SCORE, per_class=8, include=["GCSLabel"])

### 7.2 Everything the Database Knows About One Poster

One crowded poster at the centre: its labels, the face and object boxes, and its embedding (if you ran `embed.py`). `depth=2` would also reach the posters that share its labels.

In [ ]:
response, _ = run(queries.crowded_posters(min_faces=1, dataset=DATASET, limit=1))
poster = table(response).gcs_object_name.iloc[0]
poster

In [ ]:
gv.render_neighborhood(client, "_Image", {"gcs_object_name": ["==", poster]}, depth=1, per_class=12,
                       label_props=LABELS, sort=BY_SCORE, title=poster)

## 8. What Overlaps What

**The question:** a vector cannot tell whether two things on a poster overlap; the database can. Which detected **person** boxes overlap which **face** boxes on the same poster? (Swap `"Person"` for `"Logo"` to find logos covering faces.)

*How:* `RegionIoU` computes the intersection-over-union between two sets of regions. `demo.ipynb` uses the same idea for a person holding a ball.

In [ ]:
response, _ = run(queries.overlap_candidates("Person", dataset=DATASET, limit=5))
candidates = table(response)
candidates

In [ ]:
if candidates.empty:
    print("No poster with both a Person and a face box in this dataset.")
else:
    image_id = candidates["id"].iloc[0]
    response, _ = run(queries.object_face_overlap(image_id, "Person"))
    people, faces = table(response, 1), table(response, 2)
    iou = pd.DataFrame(response[-1]["RegionIoU"]["IoU"],
                       index=[f"Person (score {s:.2f})" for s in people.score],
                       columns=[f"face (joy {j:.2f})" for j in faces.joy])
    display(iou.style.format("{:.2f}").background_gradient(cmap="Blues", axis=None))

## 9. How Sure Are the Objects?

`enrich.py` stores object proposals scoring at least `--confidence-threshold` (default **0.3**); the per-image `object_count`
and the queries above use **0.5**. Boxes in between stay in the database with their score on the `HasDetection` edge (see 6.4), so the
cut-off can move without calling Google again. `enrich_stats.json` records what the detector returned in total. The table shows how
many proposals fall in each score range, how many were stored and how many count as confident. Re-run with a different floor
(`python enrich.py --only object --replace --confidence-threshold 0.2`) to store more.

In [ ]:
path = "enrich_stats.json"
if not os.path.exists(path):
    print("Run enrich.py first - it writes", path)
else:
    stats = json.load(open(path))
    cut = stats["confidence_threshold"]
    detector = stats["detectors"].get("object")
    if not detector:
        print("The last enrich run did not process the object detector.")
    else:
        hist = pd.DataFrame({
            "score range": [f"{i / 10:.1f} - {(i + 1) / 10:.1f}" for i in range(10)],
            "proposals": detector["score_histogram"],
        })
        hist["stored"] = [n if (i + 1) / 10 > cut else 0 for i, n in enumerate(detector["score_histogram"])]
        hist["confident (>= 0.5)"] = [n if i >= 5 else 0 for i, n in enumerate(detector["score_histogram"])]
        hist["share"] = (hist.proposals / max(1, hist.proposals.sum()) * 40).round().astype(int).map(lambda n: "#" * n)
        print(f"threshold {cut}: stored {detector['kept']} of {detector['seen']} proposals "
              f"({detector['kept'] / max(1, detector['seen']):.0%}) over {detector['images']} images")
        display(hist)

## 10. Vector Search, With and Without the Metadata

Optional: needs CLIP embeddings (`pip install -r requirements-embeddings.txt`, then `python embed.py`). The search finds posters that *look like* the
ask; the filter adds what the detectors know. `demo.ipynb` shows this on one ask, `quality.ipynb` scores several.

In [ ]:
import vector_search as vs

ASK = "something a child could watch tonight, where everyone looks happy"
try:
    rows, blobs = vs.search(client, ASK, k=6, criteria=vs.family_criteria(), dataset=DATASET)
except Exception as exc:   # no embeddings yet, or torch not installed
    print("Vector search not available:", str(exc)[:200])
else:
    thumbs(blobs, [f"sim {r['similarity']:.2f}, {r['face_count']} face(s)" for r in rows])

## Try Your Own

* Edit the arguments above (`"golf"`, thresholds, ...) or write a new query function in `queries.py`.
* Every query is saved as JSON in `queries/` - paste one into the ApertureDB web UI's query editor
  (`docker compose --profile ui up -d` -> http://localhost:8088, login admin / admin).
* `python graph_viz.py node _Image --where gcs_object_stem=<name> --label _Image=gcs_object_stem --open`
  draws any poster from the command line.
* Narrow everything to one folder with `DATASET = "<bucket>/<prefix>"` in the first cell.